# TFTExplainer variable importance — iteration 4

Run from the **same folder** as notebooks 01/02 (the one containing `darts_logs/`, `series_cache/`, `column_roles.json`, `run_metadata.json`). Nothing from notebook 02 needs to be in memory.

**Run the cells in order.** Cell 1 defines the stubs the checkpoint needs to unpickle.

- Cell 1: imports and callback stubs
- Cell 2: rebuild `val_seq` / `val_cov` from the saved cache (check the printed probe line: 473 days ending 2026-09-30, covariates ending 2026-12-31)
- Cell 3: load the best checkpoint, run TFTExplainer, print tables, save CSVs and plot, stability check
- Cell 4: decoder importance for a single forecast day (`TARGET_DATE`), compared with the 92-day average
- Cell 5: diagnostics (dead inputs, counterfactuals on the 2026 forecast, in-sample Diwali-2025 backtest)

Importances are Variable Selection Network attention weights, not effect on accuracy.

## Cell 1 — imports and stubs

In [ ]:
# ----- CELL 1: imports + stubs needed to unpickle the checkpoint ---------------------
import os, sys, re, json, collections.abc
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import TFTModel
from darts.explainability import TFTExplainer
from pytorch_lightning import Callback

from tft_checks import EmbeddingGate          # noqa: F401  (pickled inside the checkpoint)


class TBDiagnostics(Callback):                # stub: the checkpoint pickles __main__.TBDiagnostics
    pass

## Cell 2 — rebuild validation series and covariates

In [ ]:
# ----- CELL 2: rebuild val_seq / val_cov from the files notebook 02 saved -------------
BASE = os.getcwd()
CACHE_DIR = os.path.join(BASE, "series_cache")
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
with open(os.path.join(BASE, "run_metadata.json")) as f:
    MODEL_NAME = json.load(f)["model_name"]

target_col        = ROLES["target_col"]
FREQ              = ROLES["freq"]
static_covariates = ROLES["static_covariates"]
binary_covariates = ROLES["binary_covariates"]
series_covariates = ROLES["series_covariates"]
future_covariates = ROLES["future_covariates"]
CALENDAR_COLS     = ROLES["calendar_covariates"]
PETROL_COLS       = ROLES["shared_continuous_covariates"]

# same festive-window list as notebook 02, Section 0
PENALTY_COLS = ([f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
                + [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]
                + ["C"] + [f"C+{i}" for i in range(1, 7)])
PENALTY_COLS = [c for c in PENALTY_COLS if c in future_covariates]
OTHER_FLAGS  = [c for c in binary_covariates if c not in PENALTY_COLS]


def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")


series_index = pd.read_parquet(os.path.join(CACHE_DIR, "series_index.parquet"))
SPEND_PATHS  = np.load(os.path.join(CACHE_DIR, "spend_paths.npy"))
SHARED_COV   = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
CAL_INDEX, CAL_VALUES = SHARED_COV.time_index, SHARED_COV.values(copy=False)
assert len(CAL_INDEX) == SPEND_PATHS.shape[1]


class TargetSeq(collections.abc.Sequence):
    def __init__(self, keys, code_rows):
        self.keys, self.code_rows, self._ram = list(keys), code_rows, {}

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        if i not in self._ram:
            with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
                sales, start = z["val_sales"], str(z["val_start"])
            self._ram[i] = TimeSeries.from_times_and_values(
                pd.date_range(start, periods=len(sales), freq=FREQ),
                sales.reshape(-1, 1).astype(np.float32), columns=[target_col],
                static_covariates=pd.DataFrame([self.code_rows[i]], columns=static_covariates,
                                               dtype=np.float32))
        return self._ram[i]


class SeriesCov(collections.abc.Sequence):
    def __init__(self, path_ids):
        self.path_ids, self._ram = np.asarray(path_ids, dtype=np.int64), {}

    def __len__(self):
        return len(self.path_ids)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        pid = int(self.path_ids[i])
        if pid not in self._ram:
            vals = np.concatenate([CAL_VALUES, SPEND_PATHS[pid]], axis=1).astype(np.float32, copy=False)
            self._ram[pid] = TimeSeries.from_times_and_values(CAL_INDEX, vals, columns=list(future_covariates))
        return self._ram[pid]


# identical selection and order to notebook 02: in_val series, shuffled with seed 42
sel = series_index[series_index["in_val"]].sample(frac=1.0, random_state=42)
val_seq = TargetSeq(sel["key"], sel[static_covariates].to_numpy())
val_cov = SeriesCov(sel["spend_path"].to_numpy())
print(f"val_seq: {len(val_seq):,} series | probe: {val_seq[0].start_time().date()} -> "
      f"{val_seq[0].end_time().date()} ({len(val_seq[0])} days) | cov ends {val_cov[0].end_time().date()}")

## Cell 3 — explain, tables, plots, stability

In [ ]:
# ----- CELL 3: load model + explain ---------------------------------------------------
N_EXPLAIN     = 1000
RUN_STABILITY = True
TOP_K_PLOT    = 25

best = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)      # needs cwd = folder with darts_logs/


def clean_name(name):
    name = str(name)
    m = re.fullmatch(r"static_covariate_(\d+)", name)
    if m:
        return static_covariates[int(m.group(1))]
    return re.sub(r"_(fut_cov|past_cov|static_cov|target|futcov|pastcov)$", "", name)


def group_of(part, var):
    if part == "static":
        return "static (embedded)"
    if var == target_col:
        return "past sales (target history)"
    if "relative_index" in var:
        return "relative index"
    if var in series_covariates:
        return "spend (per-series)"
    if var in PETROL_COLS:
        return "petrol"
    if var in CALENDAR_COLS:
        return "calendar (dow, month-end)"
    if var in PENALTY_COLS:
        return "festive-window flags"
    if var in OTHER_FLAGS:
        return "other binary flags"
    return "UNMATCHED"


def _explain_chunk(ser, cov):
    """TFTExplainer needs len(series) <= model.batch_size, so it is called on chunks."""
    explainer = TFTExplainer(best, background_series=ser, background_future_covariates=cov)
    result = explainer.explain()
    try:
        w = result.get_feature_importances()
    except AttributeError:
        w = dict(encoder_importance=result.get_encoder_importance(),
                 decoder_importance=result.get_decoder_importance(),
                 static_covariates_importance=result.get_static_covariates_importance())
    means = {}
    for key, part in [("encoder_importance", "encoder"), ("decoder_importance", "decoder"),
                      ("static_covariates_importance", "static")]:
        df = w.get(key)
        if df is None or len(df) == 0:
            continue
        means[part] = df.mean(axis=0)
    return means, result, explainer


def explain_sample(lo, hi):
    bs = best.batch_size                           # 256 for this model
    acc, n_total = {}, 0
    for a in range(lo, hi, bs):
        b = min(a + bs, hi)
        ser = [val_seq[i] for i in range(a, b)]
        cov = [val_cov[i] for i in range(a, b)]
        assert all(c.end_time() >= s_.end_time() + pd.Timedelta(days=best.output_chunk_length)
                   for s_, c in zip(ser[:20], cov[:20])), "future covariates do not reach FORECAST_END"
        means, result, explainer = _explain_chunk(ser, cov)
        n = b - a
        for part, s_ in means.items():             # size-weighted running sum of per-chunk means
            acc[part] = s_ * n if part not in acc else acc[part] + s_ * n
        n_total += n
        print(f"  explained series {a}-{b} ({n_total}/{hi - lo})")
    rows = []
    for part, total in acc.items():
        s_ = total / n_total
        s_ = s_ * (100.0 / s_.sum())               # force % within each part, whatever the raw scale
        for name, val in s_.items():
            v = clean_name(name)
            rows.append(dict(part=part, variable=v, importance_pct=float(val), group=group_of(part, v)))
    out = pd.DataFrame(rows)
    out["uniform_pct"] = 100.0 / out.groupby("part")["variable"].transform("count")
    out["x_uniform"] = out["importance_pct"] / out["uniform_pct"]
    return out, result, explainer


imp, result, explainer = explain_sample(0, N_EXPLAIN)
imp = imp.sort_values(["part", "importance_pct"], ascending=[True, False]).reset_index(drop=True)

bad = imp[imp["group"] == "UNMATCHED"]
if len(bad):
    print("*** UNMATCHED variable names (fix clean_name / group_of):", bad["variable"].tolist()[:10])
print(imp.groupby("part").size().to_dict(), "variables per part")
print(imp.groupby("part")["importance_pct"].sum().round(2).to_dict(), "(sum per part)")

wide = (imp.pivot_table(index=["group", "variable"], columns="part",
                        values="importance_pct", aggfunc="first").reset_index())
print("\n=== ALL variables (% of VSN weight within each part) ===")
print(wide.sort_values(["group", "decoder"], ascending=[True, False]).round(3).to_string(index=False))

grp = imp.groupby(["part", "group"])["importance_pct"].sum().unstack("part").fillna(0).round(2)
n_vars = imp.groupby(["part", "group"]).size().unstack("part").fillna(0).astype(int).add_prefix("n_vars_")
print("\n=== By group ===")
print(grp.join(n_vars).to_string())

imp.to_csv(os.path.join(BASE, "tft_variable_importance_long.csv"), index=False)
wide.to_csv(os.path.join(BASE, "tft_variable_importance_wide.csv"), index=False)

parts = [p for p in ["static", "encoder", "decoder"] if p in imp["part"].unique()]
fig, axes = plt.subplots(1, len(parts), figsize=(7 * len(parts), 8))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, parts):
    g = imp[imp["part"] == p].nlargest(TOP_K_PLOT, "importance_pct").sort_values("importance_pct")
    ax.barh(g["variable"], g["importance_pct"])
    ax.axvline(g["uniform_pct"].iloc[0], ls="--", c="grey", lw=1)
    ax.set_title(f"{p}: top {min(TOP_K_PLOT, len(g))}  (dashed = equal weights)")
    ax.set_xlabel("% of VSN weight")
plt.tight_layout()
plt.savefig(os.path.join(BASE, "tft_variable_importance.png"), dpi=150)
plt.show()

if RUN_STABILITY:
    imp_b, _, _ = explain_sample(N_EXPLAIN, 2 * N_EXPLAIN)
    for p in parts:
        a = imp[imp.part == p].set_index("variable")["importance_pct"]
        b = imp_b[imp_b.part == p].set_index("variable")["importance_pct"].reindex(a.index)
        print(f"rank agreement (Spearman), two disjoint samples, {p:8s}: {a.corr(b, method='spearman'):.3f}")

## Cell 4 — importance for one forecast day (2026-11-06)

In [ ]:
# ----- CELL 4: importance for ONE forecast day (2026-11-06) ----------------------------
# Needs cells 1-3 run first (uses best, val_seq, val_cov, clean_name, group_of, SHARED_COV).
TARGET_DATE   = pd.Timestamp("2026-11-06")
N_EXPLAIN_DAY = 1000
TOP_K_DAY     = 20

FC_START   = pd.Timestamp(ROLES["forecast_start"])
H0         = (TARGET_DATE - FC_START).days                      # 0-based step inside the 92-day horizon
assert 0 <= H0 < best.output_chunk_length, "TARGET_DATE is outside the forecast window"
print(f"{TARGET_DATE.date()} is forecast step {H0 + 1} of {best.output_chunk_length} "
      f"(window starts {FC_START.date()})")

# which festive / calendar flags are switched on for that day (from the shared calendar)
_cal_row = SHARED_COV.pd_dataframe().loc[TARGET_DATE]
print("binary flags = 1 on that day:", [c for c in binary_covariates if _cal_row[c] == 1.0] or "none")


def _raw_weights(ser, cov):
    explainer = TFTExplainer(best, background_series=ser, background_future_covariates=cov)
    result = explainer.explain()
    try:
        return result.get_feature_importances()
    except AttributeError:
        return dict(encoder_importance=result.get_encoder_importance(),
                    decoder_importance=result.get_decoder_importance(),
                    static_covariates_importance=result.get_static_covariates_importance())


def _day_row(df):
    """Row(s) of a decoder-importance table that belong to TARGET_DATE, averaged over samples."""
    if isinstance(df.index, pd.DatetimeIndex) and TARGET_DATE in df.index:
        return df.loc[[TARGET_DATE]].mean(axis=0)
    if len(df) == best.output_chunk_length:                     # one row per forecast step, no dates
        return df.iloc[H0]
    raise ValueError(f"Cannot locate {TARGET_DATE.date()} in the decoder importance table "
                     f"(shape {df.shape}, index head {list(df.index[:3])}). Inspect it and adapt _day_row.")


bs = best.batch_size
acc_day, acc_avg, acc_enc, acc_stat, n_tot = None, None, None, None, 0
for a in range(0, N_EXPLAIN_DAY, bs):
    b = min(a + bs, N_EXPLAIN_DAY)
    ser = [val_seq[i] for i in range(a, b)]
    cov = [val_cov[i] for i in range(a, b)]
    w = _raw_weights(ser, cov)
    dec = w["decoder_importance"]
    if a == 0:
        print("decoder importance table: shape", dec.shape, "| index head", list(dec.index[:3]))
    n = b - a
    day, avg = _day_row(dec) * n, dec.mean(axis=0) * n
    acc_day = day if acc_day is None else acc_day + day
    acc_avg = avg if acc_avg is None else acc_avg + avg
    e = w["encoder_importance"].mean(axis=0) * n
    acc_enc = e if acc_enc is None else acc_enc + e
    st_ = w.get("static_covariates_importance")
    if st_ is not None and len(st_):
        v = st_.mean(axis=0) * n
        acc_stat = v if acc_stat is None else acc_stat + v
    n_tot += n
    print(f"  explained series {a}-{b} ({n_tot}/{N_EXPLAIN_DAY})")


def _tidy(total, part):
    s_ = total / n_tot
    s_ = s_ * (100.0 / s_.sum())
    return pd.DataFrame({"part": part, "variable": [clean_name(x) for x in s_.index],
                         "importance_pct": s_.values})


day_df = _tidy(acc_day, "decoder (day)")
avg_df = _tidy(acc_avg, "decoder (all 92 days)")
dec_cmp = day_df.merge(avg_df[["variable", "importance_pct"]], on="variable", suffixes=("_day", "_avg"))
dec_cmp["group"] = [group_of("decoder", v) for v in dec_cmp["variable"]]
dec_cmp["x_uniform_day"] = dec_cmp["importance_pct_day"] / (100.0 / len(dec_cmp))
dec_cmp["day_minus_avg"] = dec_cmp["importance_pct_day"] - dec_cmp["importance_pct_avg"]
dec_cmp = dec_cmp.drop(columns="part").sort_values("importance_pct_day", ascending=False)

print(f"\n=== DECODER, {TARGET_DATE.date()} (top {TOP_K_DAY}); uniform = {100.0 / len(dec_cmp):.2f}% ===")
print(dec_cmp.head(TOP_K_DAY).round(3).to_string(index=False))
print("\n=== DECODER by group: this day vs average over the 92 days (% summed) ===")
print(dec_cmp.groupby("group")[["importance_pct_day", "importance_pct_avg"]].sum().round(2)
      .sort_values("importance_pct_day", ascending=False).to_string())

enc_df = _tidy(acc_enc, "encoder")
enc_df["group"] = [group_of("encoder", v) for v in enc_df["variable"]]
print("\n=== ENCODER (window-level; same for every forecast day) top 10 ===")
print(enc_df.sort_values("importance_pct", ascending=False).head(10).round(3).to_string(index=False))
if acc_stat is not None:
    st_df = _tidy(acc_stat, "static")
    print("\n=== STATIC (same for every forecast day) ===")
    print(st_df.sort_values("importance_pct", ascending=False).round(3).to_string(index=False))

dec_cmp.to_csv(os.path.join(BASE, f"tft_decoder_importance_{TARGET_DATE.date()}.csv"), index=False)

## Cell 5 — diagnostics: why was 2026-11-06 under-predicted?

In [ ]:
# ----- CELL 5: why was 2026-11-06 under-predicted? diagnostics --------------------------
# Needs cells 1-4 run first (uses best, val_seq, val_cov, SHARED_COV, imp, TARGET_DATE, H0, ...).
N_DIAG    = 256            # series per diagnostic (kept <= batch size)
N_SAMPLES = 100            # NegativeBinomial samples per series
FC_START  = pd.Timestamp(ROLES["forecast_start"])
TARGET_DATE = pd.Timestamp("2026-11-06")
H0 = (TARGET_DATE - FC_START).days
cal_df = SHARED_COV.pd_dataframe()

# ---- D1: decoder weight sitting on flags that never fire in Oct-Dec 2026 ---------------
fc_cal = cal_df.loc[ROLES["forecast_start"]:ROLES["forecast_end"]]
dead = [c for c in binary_covariates if fc_cal[c].max() == 0]
d = imp[imp.part == "decoder"]
print(f"D1: {len(dead)} of {len(binary_covariates)} binary flags never fire in the forecast window; "
      f"they hold {d[d.variable.isin(dead)].importance_pct.sum():.1f}% of decoder VSN weight")
print(d[d.variable.isin(dead)].nlargest(8, "importance_pct")[["variable", "importance_pct"]].round(2).to_string(index=False))
print("D-2 flag fires on:", [x.date().isoformat() for x in cal_df.index[cal_df["D-2"] == 1]])
print("   (each training year contributes ONE D-2 day per series)\n")


# ---- helpers ---------------------------------------------------------------------------
def predict_samples(ser, cov, n=None, ns=N_SAMPLES):
    n = n or best.output_chunk_length
    preds = best.predict(n=n, series=ser, future_covariates=cov, num_samples=ns, verbose=False)
    vals = np.stack([p.all_values(copy=False)[:, 0, :] for p in preds])      # (series, days, samples)
    return preds[0].time_index, vals


def edit_cov(cov, start, edits):
    cols = list(cov.components)
    vals = cov.values().copy()
    i0 = int((start - cov.start_time()).days)
    for c, v in edits.items():
        vals[i0:, cols.index(c)] = v
    return TimeSeries.from_times_and_values(cov.time_index, vals, columns=cols)


_cache = {}
def edited(covs, edits, tag):
    out = []
    for c in covs:
        k = (tag, id(c))
        if k not in _cache:
            _cache[k] = edit_cov(c, FC_START, edits)
        out.append(_cache[k])
    return out


ser = [val_seq[i] for i in range(N_DIAG)]
covs = [val_cov[i] for i in range(N_DIAG)]
assert len(ser) <= best.batch_size

# ---- D2: counterfactuals on the 2026 forecast ----------------------------------------------
z_petrol_2025 = float(cal_df.loc[pd.Timestamp("2025-10-18"), "PETROL_PRICE_INDEX"])
scenarios = {
    "baseline": covs,
    "petrol -> Oct-2025 level": edited(covs, {"PETROL_PRICE_INDEX": z_petrol_2025}, "p25"),
    "petrol -> training mean": edited(covs, {"PETROL_PRICE_INDEX": 0.0}, "p0"),
    "D-2 flag off": edited(covs, {"D-2": 0.0}, "d2off"),
    "all N/D/C window flags off": edited(covs, {c: 0.0 for c in PENALTY_COLS}, "festoff"),
}
rows = []
for name, cv in scenarios.items():
    _, v = predict_samples(ser, cv)
    mean_day = v.mean(axis=2).sum(axis=0)                  # summed over the N_DIAG series, per day
    med_day = np.median(v, axis=2).sum(axis=0)
    rows.append(dict(scenario=name, nov6_sum_of_means=mean_day[H0], nov6_sum_of_medians=med_day[H0],
                     uplift_vs_window_avg=mean_day[H0] / mean_day.mean()))
res = pd.DataFrame(rows)
res["vs_baseline"] = res["nov6_sum_of_means"] / res.loc[0, "nov6_sum_of_means"]
print(f"D2: forecast for {TARGET_DATE.date()} on {N_DIAG} series, scenario by scenario")
print(res.round(3).to_string(index=False))
print("   sum_of_medians << sum_of_means => sparse counts: point forecasts taken as the median understate totals\n")

# ---- D3: in-sample backtest of Diwali 2025 (model was trained on this period) -----------------
def train_series(i):
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(val_seq.keys[i])}.npz")) as z:
        sales, start = z["train_sales"], str(z["train_start"])
    return TimeSeries.from_times_and_values(
        pd.date_range(start, periods=len(sales), freq=FREQ), sales.reshape(-1, 1).astype(np.float32),
        columns=[target_col], static_covariates=val_seq[i].static_covariates)


CUT = pd.Timestamp("2025-09-30")
full_ts = [train_series(i) for i in range(N_DIAG)]
hist = [t.slice(CUT - pd.Timedelta(days=best.input_chunk_length - 1), CUT) for t in full_ts]
idx25, v25 = predict_samples(hist, covs)
assert idx25[0] == CUT + pd.Timedelta(days=1)
pred25 = v25.mean(axis=2).sum(axis=0)
act25 = np.sum([t.slice(CUT + pd.Timedelta(days=1), CUT + pd.Timedelta(days=best.output_chunk_length)).values()[:, 0]
                for t in full_ts], axis=0)
h25 = (pd.Timestamp("2025-10-18") - (CUT + pd.Timedelta(days=1))).days
print("D3: in-sample backtest, window 2025-10-01 -> 2025-12-31 (same series, same model)")
print(f"   actual    D-2 day {act25[h25]:9.0f} | uplift vs window avg {act25[h25] / act25.mean():.2f}x")
print(f"   predicted D-2 day {pred25[h25]:9.0f} | uplift vs window avg {pred25[h25] / pred25.mean():.2f}x")
print(f"   whole-window predicted / actual: {pred25.sum() / act25.sum():.2f}")
print("   If the peak is also missed here, the model cannot fit it at all (training budget / capacity / loss).")
print("   If it is caught here but missed for 2026, the failure is generalisation (covariate shift, rare flag).")